# CIDS · 20 — out-of-fold SegResNet probability maps (for the boundary refiner)

**Why:** the refiner needs SegResNet's probability map as an extra input (where the finder was unsure). Our SegResNet-100 was trained on all 1080 training patients, so its maps on those patients are too good: it has seen their labels. Here the training patients are split into **3 folds**. Each fold model trains on the other two folds (~720 patients) and predicts the **held-out** ~360. Every training patient then gets an honest map from a model that never saw it.

**Run this notebook 4 times, changing only `FOLD` in the first cell:**

| `FOLD` | What it does | Time | Inputs |
|---|---|---|---|
| `0`, `1`, `2` | trains SegResNet 100 epochs on 2/3 of training patients (val for best epoch), then saves maps of the held-out third | ~10 h each | cache |
| `'full'` | no training: the existing SegResNet-100 saves maps for val (120) + test (306) | ~1 h | cache + `cids-checkpoints` |

Folds are independent: run them on both accounts in parallel. Output per fold: `heldout_pred/<case>.npz` (`mask`, `prob` = probability × 255, uint8) + `heldout_metrics.csv`.
If a fold stops at the time budget, it still saves the maps from its best epoch. To continue it instead, attach its output and set `RESUME_FROM`.

**Settings:** GPU T4 x2 · Internet ON. Run with **Save Version → Save & Run All**.


In [ ]:
FOLD = 0                      # 0, 1, 2  or  'full'

REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
SEG_CKPT   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints/segresnet_img_e100_best.pt'   # FOLD='full' only
EPOCHS        = 100
TIME_BUDGET_H = 10.0          # training stops before this; the held-out maps are still saved (~30 min)
RESUME_FROM   = ''            # last.pt of an unfinished run of the SAME fold (attached output), to continue
CODE_DIR = '/kaggle/working/cids'
OUT      = f'/kaggle/working/seg_fold{FOLD}'
PATHS = [CACHE_ROOT] + ([SEG_CKPT] if FOLD == 'full' else [])


In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not)."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')

In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'python -m pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)
assert FOLD in (0, 1, 2, 'full'), FOLD
import pandas as pd
sp = pd.read_csv('configs/splits.csv')
if FOLD != 'full':
    print(sp[f'fold3_{FOLD}'].value_counts().to_dict())


## Smoke test (~3 min): 6 training cases, 1 epoch, 3 cases evaluated


In [ ]:
if FOLD == 'full':
    sh(f'mkdir -p /kaggle/working/seg_smoke && cp {SEG_CKPT} /kaggle/working/seg_smoke/best.pt')
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out /kaggle/working/seg_smoke --eval_only --eval_split val,test --save_prob --limit_eval 3')
else:
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out /kaggle/working/seg_smoke --split_col fold3_{FOLD} --eval_split heldout --save_prob --epochs 1 --val_every 1 --limit_train 6 --limit_eval 3')
sh('ls /kaggle/working/seg_smoke && rm -rf /kaggle/working/seg_smoke')


## Full run
**Read:** for a fold, the val Dice every 5 epochs (the full model reached ~0.72 at 100 epochs; a fold model on 2/3 of the data should land a little lower), then `HELDOUT (~360 cases)`. That is the honest Dice on patients this model never saw. For `'full'`: `VAL` and `TEST` should be close to notebook 02's raw-mask numbers (test raw Dice was ~0.64–0.67; our reported 0.684 is after rel20 post-processing).


In [ ]:
if FOLD == 'full':
    sh(f'mkdir -p {OUT} && cp {SEG_CKPT} {OUT}/best.pt')
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out {OUT} --eval_only --eval_split val,test --save_prob')
else:
    resume = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''
    sh(f'python scripts/s1_train_segresnet.py --cache_root {CACHE_ROOT} --out {OUT} --split_col fold3_{FOLD} --eval_split heldout --save_prob --epochs {EPOCHS} --time_budget_h {TIME_BUDGET_H} {resume}')


In [ ]:
import json, glob
for f in sorted(glob.glob(f'{OUT}/*_summary.json')):
    print(f, json.dumps(json.load(open(f)), indent=1))
for d in sorted(glob.glob(f'{OUT}/*_pred')):
    print(d, len(glob.glob(d + '/*.npz')), 'maps')
sh(f'du -sh {OUT}')
